# Unifying Quantum ML with Keras 3.0 and PennyLane

**Goal:** Showcase how to build, train, and deploy multi-backend hybrid QML models using the new Keras 3-compatible PennyLane plugin.

## 1. Introduction: High-Level Hybrid QML with Keras 3

Historically, the integration of quantum circuits into classical deep learning workflows was constrained by framework-specific limitations. Researchers utilizing the Keras API were primarily restricted to the TensorFlow ecosystem, while those using PennyLane were required to maintain separate, backend-specific implementations for different deep learning frameworks.

The emergence of **Keras 3.0**, in conjunction with this unified PennyLane integration, represents a significant paradigm shift. By leveraging the multi-backend capabilities of Keras 3.0, hybrid quantum-classical models can now achieve complete portability across **JAX, PyTorch, and TensorFlow** through a single, standardized interface.

### Setup & Imports

The first step is to configure our backend. Keras 3.0 allows you to choose your favorite engine before any imports occur.

In [ ]:
import os

# Select your backend: "jax", "torch", or "tensorflow"
# os.environ["KERAS_BACKEND"] = "jax"

import keras
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from pennylane_keras_layer import KerasCircuitLayer

print(f"--- Running on Keras Backend: {keras.backend.backend()} ---")

## 2. Step 1: Defining the Quantum QNode

PennyLane's `QNode` is the perfect way to define variational quantum circuits. We'll start with a simple circuit that uses `AngleEmbedding` for data input and `StronglyEntanglingLayers` for trainable parameters.

In [ ]:
n_qubits = 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def qnode(weights, inputs):
    # Encoding classical data into quantum states
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    
    # Trainable quantum layers
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    
    # Measuring the result (Expval of PauliZ on each qubit)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Specify the shapes for the trainable parameters
weight_shapes = {"weights": (2, n_qubits, 3)}

## 3. Step 2 & 3: The Keras 3.0 Hybrid Workflow

The beauty of the `KerasCircuitLayer` is its simplicity. The workflow is exactly the same as any other Keras model:
1. **Wrap** the QNode.
2. **Assemble** in a `Sequential` or `Functional` API.
3. **Fit** using standard classical optimizers.

In [ ]:
# 1. Wrap the QNode in a Keras-native layer
qlayer = KerasCircuitLayer(qnode, weight_shapes, output_dim=n_qubits)

# 2. Assemble high-level components
model = keras.Sequential([
    keras.Input(shape=(n_qubits,)),
    qlayer,
    keras.layers.Dense(1, activation="sigmoid")
])

# 3. Compile and prepare data
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

X, y = make_moons(n_samples=200, noise=0.1, random_state=42)
X = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0)) * 2 - 1 # Normalize to [-1, 1]
y = y.reshape(-1, 1).astype("float32")

# Training is now a single standard call
print("Training hybrid model...")
# model.fit(X, y, epochs=10, batch_size=8, verbose=1)

## 4. Multi-Backend Power: The Deep Dive

Integrating with Keras 3.0 doesn't mean giving up on the unique features of your backend. You can take the *exact same model* and run it through a pure JAX or PyTorch optimization loop without changing a single line of your quantum circuit.

In [ ]:
backend = keras.backend.backend()

if backend == "jax":
    import jax
    import jax.numpy as jnp
    print("Example: JAX functional optimization step")
    def jax_loss_fn(params, x, y):
        logits = model.stateless_call(params, x)
        return jnp.mean((logits - y)**2)
    # grads = jax.grad(jax_loss_fn)(model.trainable_variables, X[:1], y[:1])
    
elif backend == "torch":
    import torch
    print("Example: PyTorch dynamic optimization loop")
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    # optimizer.zero_grad(); out = model(torch.tensor(X[:1])); loss.backward(); optimizer.step()

## 5. Results: Visualizing the Decision Boundary

A simple 2D classification task allows us to see exactly how the quantum circuit is shaping the classical feature space. By plotting the decision boundary, we can witness the hybrid model in action.

In [ ]:
print("Visualization snippet ready.")
# xx, yy = np.meshgrid(np.linspace(-1.1, 1.1, 20), np.linspace(-1.1, 1.1, 20))
# grid = np.c_[xx.ravel(), yy.ravel()]
# preds = model.predict(grid).reshape(xx.shape)

# plt.figure(figsize=(8, 6))
# plt.contourf(xx, yy, preds, alpha=0.8, cmap="RdBu")
# plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), edgecolors='k', cmap="RdBu")
# plt.title("Quantum-Classical Hybrid Decision Boundary")
# plt.show()